# MRIxFields Etapa 2 — Latent bank + transporte (OT-CFM → Schrödinger bridge) en A100

Corre **de arriba a abajo** en un runtime **A100** (Colab Pro+). Etapas:
1. Setup (repo `main` + deps + GPU).
2. Rutas (EDITA la celda de config).
3. Split (remapea `split_v3.json` del box a las rutas de Colab — misma membership).
4. **Validación de estrategia de encode (⚠️ la preocupación de las métricas).** El VAE usa
   **GroupNorm**, cuya normalización depende del tamaño espacial de la entrada. Entrenó y evaluó
   con parches **64³**, así que el encode full-volume 364³ puede **degradar la reconstrucción**.
   Esta sección mide SSIM3D de round-trip para full / tiled-256 / tiled-128 / tiled-64 y elige la
   que reproduce el techo ep15 (~0.973). **No armamos el bank hasta que una estrategia lo cumpla.**
5. Build del latent bank (con la estrategia ganadora).
6. Sanity de FM (mide sec/step) → entrenamiento FM (OT-CFM).
7. Entrenamiento SB (Schrödinger bridge) — misma red, solo cambia el config.
8. Eval board + submission.

**Requisitos que Tafoya sube a Drive antes de correr:**
- El checkpoint del VAE congelado `vae_kl_vae_best.pt` (~60 MB).
- El split `split_v3.json` (del box, `D:\MRI_Field_2026\split_v3.json`).
- Los datos oficiales MRIxFields (estructura `Data/Training_retrospective/...`, `Training_prospective/...`).


## 1. Setup — repo, deps, GPU


In [ ]:
!nvidia-smi
import torch
print("torch", torch.__version__, "cuda", torch.cuda.is_available())
assert torch.cuda.is_available(), "Activa un runtime A100 (Runtime > Change runtime type > A100)."
print("GPU:", torch.cuda.get_device_name(0), "VRAM GB:", round(torch.cuda.get_device_properties(0).total_memory/1e9, 1))


In [ ]:
# Clona main (ya tiene todo el código de Etapa 2) e instala en editable con los extras de datos.
%cd /content
!rm -rf MRIxFields
!git clone --depth 1 https://github.com/GuillermoTafoya/MRIxFields.git
%cd /content/MRIxFields
!pip -q install -e ".[nifti,evaluation]" scipy
# Verifica que el CLI y los comandos de Etapa 2 existen:
!python -m fieldbridge.cli build-latent-bank --help | head -5
!python -m fieldbridge.cli train-stage2-transport --help | head -5


## 2. Config — EDITA ESTAS RUTAS


In [ ]:
import os
from pathlib import Path

# --- monta Drive (donde subiste checkpoint + split + datos) ---
from google.colab import drive
drive.mount('/content/drive')

# --- EDITA: apunta a tus archivos en Drive ---
DRIVE      = "/content/drive/MyDrive/MRI_Field_2026"          # carpeta raíz en tu Drive
DATA_ROOT  = f"{DRIVE}/Data"                                   # datos oficiales (Training_retrospective/...)
CKPT       = f"{DRIVE}/vae_kl_vae_best.pt"                     # VAE congelado
SPLIT_SRC  = f"{DRIVE}/split_v3.json"                          # split del box (rutas Windows adentro)
OLD_DATA_ROOT = r"D:\MRI_Field_2026\Data"                      # raíz de datos EN EL BOX (para remapear)

# --- rutas de trabajo (efímeras del runtime) ---
WORK       = "/content/work"
LATENTS    = f"{WORK}/latents"
SPLIT      = f"{WORK}/split_colab.json"
VAE_CONFIG = "configs/experiment/stage1_vae_v2_fgw_freebits.yaml"
os.makedirs(WORK, exist_ok=True)
os.environ["MRIXFIELDS_DATA_ROOT"] = DATA_ROOT
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"  # A100/Linux: OK (no como el box Windows)

for p in (CKPT, SPLIT_SRC):
    assert Path(p).exists(), f"No existe: {p} (súbelo a Drive)"
assert Path(DATA_ROOT).is_dir(), f"No existe el data root: {DATA_ROOT}"
print("OK rutas.")


## 3. Split — remapea `split_v3.json` (box → Colab), misma membership

Reescribe solo el prefijo de ruta de cada record (Windows → Colab) y re-guarda con fingerprints
frescos. Los `case_id`/sujetos NO cambian → el train/val/test es idéntico al del box (1590/189/205).


In [ ]:
import json
from pathlib import Path
from fieldbridge.data.manifests import record_from_mapping
from fieldbridge.data.vae_splits import VaeSplits, save_vae_splits, load_vae_splits

raw = json.loads(Path(SPLIT_SRC).read_text(encoding="utf-8"))
old = OLD_DATA_ROOT.replace("\\", "/").rstrip("/")

def remap(p):
    p = str(p).replace("\\", "/")
    return p.replace(old, DATA_ROOT.rstrip("/"))

missing = 0
for split in ("train", "validation", "test"):
    for r in raw["splits"][split]:
        r["image_path"] = remap(r["image_path"])
        if not Path(r["image_path"]).exists():
            missing += 1
print("records con ruta inexistente tras remapear:", missing, "(debe ser 0)")
assert missing == 0, "El remapeo no encontró archivos: revisa DATA_ROOT / OLD_DATA_ROOT / estructura."

splits = VaeSplits(
    train=tuple(record_from_mapping(r) for r in raw["splits"]["train"]),
    validation=tuple(record_from_mapping(r) for r in raw["splits"]["validation"]),
    test=tuple(record_from_mapping(r) for r in raw["splits"]["test"]),
    seed=int(raw["seed"]), fractions=tuple(float(x) for x in raw["fractions"]),
    metadata=dict(raw.get("metadata", {})),
)
save_vae_splits(splits, SPLIT)
chk = load_vae_splits(SPLIT)
print(f"split_colab OK: train={len(chk.train)} val={len(chk.validation)} test={len(chk.test)}")


## 4. ⚠️ Validación de estrategia de encode (la preocupación de las métricas)

Mide SSIM3D de round-trip (encode → decode del latente almacenado) por estrategia, sobre unos pocos
volúmenes de test. **Elige la estrategia cuyo SSIM3D se acerque más al techo ep15 (~0.973).** Ojo:
como el VAE entrenó en 64³, es probable que **tiled-64/128 gane y full-volume pierda** — justamente
el efecto GroupNorm. En A100 el costo de más bloques es barato, así que priorizamos fidelidad de métrica.


In [ ]:
import time, torch
from pathlib import Path
from fieldbridge.cli import _load_optional_config, _model_config, _kl_vae_kwargs
from fieldbridge.models.factory import build_encoder, build_decoder
from fieldbridge.training.checkpoints import load_checkpoint
from fieldbridge.data.vae_splits import load_vae_splits
from fieldbridge.data.latent_bank import encode_latent, decode_latent_tiled, load_volume, downsample_factor
from fieldbridge.evaluation.metrics import ssim3d

cfg = _load_optional_config(Path(VAE_CONFIG)); mc = _model_config(cfg)
enc = build_encoder("kl_vae", **_kl_vae_kwargs(mc, "encoder"))
dec = build_decoder("kl_vae", **_kl_vae_kwargs(mc, "decoder"))
st = load_checkpoint(CKPT); enc.load_state_dict(st["encoder"]); dec.load_state_dict(st["decoder"])
enc.requires_grad_(False).eval(); dec.requires_grad_(False).eval()
dev = torch.device("cuda"); enc.to(dev); dec.to(dev); F = downsample_factor(enc)

N_AB = 6
recs = load_vae_splits(SPLIT).records_for("test")[:N_AB]
# (nombre, strategy, block, halo, precision)
STRATS = [
    ("full",     "full",  (1,1,1),       (0,0,0),    "bfloat16"),
    ("tiled256", "tiled", (256,256,256), (16,16,16), "bfloat16"),
    ("tiled128", "tiled", (128,128,128), (16,16,16), "bfloat16"),
    ("tiled64",  "tiled", (64,64,64),    (16,16,16), "bfloat16"),
]
results = {}
for name, strat, block, halo, prec in STRATS:
    ss, t0, ok = [], time.time(), True
    for rec in recs:
        vol = load_volume(rec).to(dev)
        try:
            lat, _ = encode_latent(enc, vol, rec.domain, strategy=strat, block_size=block, halo=halo, precision=prec)
        except RuntimeError as e:
            print(f"{name}: FALLÓ ({str(e)[:60]})"); ok = False; del vol; torch.cuda.empty_cache(); break
        recon = decode_latent_tiled(dec, lat, rec.domain, factor=F, block_size=(128,128,128), halo=(16,16,16), precision=prec)
        ss.append(float(ssim3d(recon, vol, data_range=1.0)))
        del vol, lat, recon; torch.cuda.empty_cache()
    if ok and ss:
        results[name] = (sum(ss)/len(ss), (time.time()-t0)/len(ss), (block, halo, prec, strat))
        print(f"{name:9s} mean_ssim3d={results[name][0]:.4f}  ~{results[name][1]:.1f}s/vol  (target ~0.973)")

best = max(results, key=lambda k: results[k][0])
print(f"\\n>>> Mejor por SSIM3D: {best} = {results[best][0]:.4f}")
_, _, (BLOCK, HALO, PRECISION, STRATEGY) = results[best]
print("Usaré:", dict(strategy=STRATEGY, block=BLOCK, halo=HALO, precision=PRECISION))


> Si el mejor SSIM3D queda **muy por debajo de ~0.97**, no armes el bank: avísale a Simón (algo
> cambió en el VAE o en los datos). Si `tiled64` gana pero es lento, `tiled128` suele ser buen
> compromiso; puedes forzar la estrategia editando `STRATEGY/BLOCK/HALO/PRECISION` abajo.


## 5. Build del latent bank (estrategia validada)


In [ ]:
# Usa la estrategia elegida arriba. store-dtype float16 (~7 MB/vol, ~14 GB total para 1984 vols).
bs = " ".join(map(str, BLOCK)); hs = " ".join(map(str, HALO))
cmd = (
  f"python -u -m fieldbridge.cli build-latent-bank "
  f"--config {VAE_CONFIG} --split-json {SPLIT} --checkpoint {CKPT} --out {LATENTS} "
  f"--device cuda --strategy {STRATEGY} --block-size {bs} --halo {hs} "
  f"--precision {PRECISION} --store-dtype float16 --roundtrip-samples 8"
)
print(cmd)
!{cmd}
# Guarda el bank a Drive para no re-encodear si se cae el runtime (opcional, ~14 GB):
# !cp -r {LATENTS} {DRIVE}/latents


In [ ]:
import json
m = json.load(open(f"{LATENTS}/latent_bank_manifest.json"))
print("counts:", m["counts"])
print("latent std (train):", round(m["latent_stats"]["global_std"], 4), "(esperado ~0.70)")
print("roundtrip mean SSIM3D:", m["roundtrip"]["mean_ssim3d"])


## 6. Sanity de FM (mide sec/step) — 200 steps


In [ ]:
# Sanity corto: confirma que la loss baja y mide el sec/step real antes del run largo.
!python -m fieldbridge.cli train-stage2-transport \
  --config configs/experiment/stage2_transport_fm_v1.yaml \
  --bank-dir {LATENTS} --checkpoint-dir {WORK}/fm_sanity/ckpt \
  --steps 200 --device cuda --json


## 7. Entrenamiento FM (OT-CFM) — run completo

`checkpoint-dir` a Drive para sobrevivir desconexiones (resume con `--resume-from <last>`).
Ajusta `--steps` según el sec/step del sanity y tu presupuesto de GPU.


In [ ]:
!python -u -m fieldbridge.cli train-stage2-transport \
  --config configs/experiment/stage2_transport_fm_v1.yaml \
  --bank-dir {LATENTS} --checkpoint-dir {DRIVE}/ckpt/fm_ot_cfm \
  --val --device cuda --json


## 8. Entrenamiento SB (Schrödinger bridge) — misma red, otro config

Mismo `v_θ`; solo cambia `bridge: schrodinger` (+ `sigma`) y el coupling. Empieza fresco (o
warm-start del FM copiando su `transport_..._last.pt` como `--resume-from`).


In [ ]:
!python -u -m fieldbridge.cli train-stage2-transport \
  --config configs/experiment/stage2_transport_sb_v1.yaml \
  --bank-dir {LATENTS} --checkpoint-dir {DRIVE}/ckpt/sb_brownian \
  --val --device cuda --json


## 9. Siguiente: eval board + submission

- **Eval board (métrica oficial + rank-sum):** genera predicciones sobre nuestro test integrando el
  ODE/SDE del transporte, decodifica con el VAE, y corre
  `python -m fieldbridge.cli mrixfields2026-task3-board --pred-dir <tree> --target-dir <tree>`
  (o `--summaries-json` para rankear FM vs SB). *(El sampler de inferencia ODE/SDE + el armado del
  árbol de predicciones es el próximo entregable de código — avísame y lo agrego.)*
- **Submission:** `python -m fieldbridge.cli mrixfields2026-zip-submission --submission-root <dir> --task task3 --out task3.zip`.

> Guarda los checkpoints de FM/SB (y opcionalmente el bank) a Drive: el runtime de Colab es efímero.
